# HKO Station Stacking V20 GFS-Only No Peak

Hong Kong adaptation of the KDAL V20 no-peak experiment. It uses the official HKO daily maximum,
official HKO Headquarters observations through 11 AM Hong Kong time, and exact prior-day 18Z GFS forecasts.
The model remains Fahrenheit-native for compatibility and reports both Fahrenheit and Celsius results.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "hong_kong_11am.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/hong_kong_11am.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data" / "calibration" / "hong_kong_11am"
STATION_ID = "HKO"
PROVIDERS = ("gfs",)
TIMING_MODE = "hong_kong_same_day_11am_live_safe"
FEATURE_VERSION = "v20_hko_gfs_no_peak"
TARGET_SOURCE = "hko_daily_max"
FAST_MODE = False
OPTUNA_TRIALS = 30
MODEL_VERSION = "station_high_regressor_v20_hko_gfs_no_peak_stack"
EXPORT_MODEL_WEIGHTS = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    TARGET_MODE_REMAINING_WARMUP,
    TRAINING_PROFILE_V20_ALIGNED,
    V20_EXPANDING_FOLDS,
    V20_HKO_GFS_NO_PEAK_DROPPED_FEATURE_COLUMNS,
    _fit_feature_columns,
    _modeling_frame,
    missing_model_dependencies,
)
from src.hong_kong_11am import (
    GFS_ALLOWED_GAP_END_DATE,
    GFS_USABLE_START_DATE,
    MODEL_PROVIDERS,
    OBSERVATION_SOURCE_CONTRACT,
    add_celsius_metric_columns,
    add_celsius_prediction_columns,
    hong_kong_stacking_config,
    provider_modeling_coverage,
    run_hong_kong_year_split_experiment,
)


## Contract

- HKO official daily maximum target (`hko_daily_max`)
- HKO Headquarters 1-minute temperature, maximum-since-midnight, and humidity snapshots by 11 AM
- Historical snapshots come from the free DATA.GOV.HK archive; live snapshots come from HKO Open Data
- Same-station quality contract: HKO high-so-far cannot exceed the official HKO daily maximum
- GFS-only forecast roster
- Known access-blocked gap through 2021-03-23; uninterrupted exact GFS required from 2021-03-24
- V20 expanding validation folds for 2022–2025 and 2026 out-of-fold testing
- No HRRR/NBM peak-timing feature family


In [3]:
config = hong_kong_stacking_config(
    PROJECT_ROOT,
    DATA_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
)
assert config.observation_target_same_station is True
assert config.observation_source == OBSERVATION_SOURCE_CONTRACT

fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
            "validation_weight": config.effective_year_split_validation_weights[fold.validation_year],
        }
        for fold in V20_EXPANDING_FOLDS
    ]
)
fold_spec


,fold,train_start_year,train_end_year,validation_year,validation_weight
0,fold_2021_to_2022,2021,2021,2022,1.0
1,fold_2021_2022_to_2023,2021,2022,2023,1.0
2,fold_2021_2023_to_2024,2021,2023,2024,1.0
3,fold_2021_2024_to_2025,2021,2024,2025,1.0


## Data Readiness


In [4]:
coverage = pd.DataFrame(
    [provider_modeling_coverage(DATA_ROOT, provider) for provider in MODEL_PROVIDERS]
)
coverage[
    [
        "provider",
        "usable_start_date",
        "usable_end_date",
        "ok_rows",
        "required_usable_rows",
        "allowed_early_gap_rows",
        "modeling_ready",
    ]
]


,provider,usable_start_date,usable_end_date,ok_rows,required_usable_rows,allowed_early_gap_rows,modeling_ready
0,gfs,2021-03-24,2026-07-20,1945,1945,82,True


## Train and Score


In [5]:
missing_packages = missing_model_dependencies(config.effective_base_model_methods)
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

if not coverage["modeling_ready"].all():
    raise RuntimeError("GFS coverage is not ready; inspect missing_usable_dates before training")

result = run_hong_kong_year_split_experiment(
    DATA_ROOT,
    project_root=PROJECT_ROOT,
    providers=MODEL_PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
)
result.scoreboard


[I 2026-07-26 11:43:09,883] A new study created in RDB with name: HKO_v20_hko_gfs_no_peak_remaining_warmup_v20_aligned_base_xgboost_mae_f_wide_obs_hko_open_data_archive_1min
[I 2026-07-26 11:44:01,142] Trial 0 finished with value: 1.2422631061604168 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.2422631061604168.
[I 2026-07-26 11:44:14,661] Trial 1 finished with value: 1.4131689285002262 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 w

,period,method,count,mae_f,rmse_f
0,validation_2022_2025,xgboost,1447,1.162343,1.471225
1,validation_2022_2025,lightgbm,1447,1.174450,1.502566
2,validation_2022_2025,catboost,1447,1.155186,1.463264
3,validation_2022_2025,gfs_raw,1447,4.371660,4.960448
4,test_2026,xgboost,201,1.128274,1.435054
5,test_2026,lightgbm,201,1.112866,1.422768
6,test_2026,catboost,201,1.130054,1.415981
7,test_2026,ridge_stack,201,1.115964,1.403891
8,test_2026,gfs_raw,201,4.450877,4.976509


## Export Complete Base + Ridge Bundle


In [6]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=TIMING_MODE,
        providers=MODEL_PROVIDERS,
        feature_version=FEATURE_VERSION,
        training_profile=TRAINING_PROFILE_V20_ALIGNED,
        optuna_metric="mae_f",
        target_mode=TARGET_MODE_REMAINING_WARMUP,
        target_source=TARGET_SOURCE,
        base_model_methods=("xgboost", "lightgbm", "catboost"),
        stack_enabled=True,
        source_pipeline="notebooks/experiments/station_stacking_v20_hko_no_peak",
        max_feature_missing_fraction=0.03,
        bucket_contract="floor_1c",
        observation_target_same_station=True,
        observation_source=OBSERVATION_SOURCE_CONTRACT,
    )
    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled.")


## Included and Pruned Feature Audit


In [7]:
included_features = result.feature_columns.sort_values(["kind", "feature"]).reset_index(drop=True)
pruned_single_provider_features = pd.DataFrame(
    {"feature": sorted(V20_HKO_GFS_NO_PEAK_DROPPED_FEATURE_COLUMNS)}
)
included_features, pruned_single_provider_features


(                                             feature         kind
 0                                        day_of_week  categorical
 1                          observed_precip_intensity  categorical
 2                     observed_weather_code_at_as_of  categorical
 3                                 actual_high_lag_1d      numeric
 4                                 actual_high_lag_2d      numeric
 ..                                               ...          ...
 116     v8_provider_mean_remaining_vs_month_normal_f      numeric
 117            v8_recent_remaining_warmup_30d_mean_f      numeric
 118             v8_recent_remaining_warmup_7d_mean_f      numeric
 119    v8_wind_gust_max_remaining_warmup_interaction      numeric
 120  v8_wind_speed_mean_remaining_warmup_interaction      numeric
 
 [121 rows x 2 columns],
                                             feature
 0                                     actual_high_c
 1    actual_minus_climatology_10y_f_DIAGNOSTIC_ONLY
 2        

In [8]:
feature_coverage = (
    result.features[included_features["feature"].tolist()]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
feature_coverage


,feature,coverage_pct
0,day_of_week,100.000000
1,observed_precip_intensity,100.000000
2,actual_high_lag_1d,100.000000
3,actual_high_lag_2d,100.000000
4,actual_high_lag_3d,100.000000
...,...,...
116,gfs_prior_month_mae_f,94.770597
117,gfs_prior_month_bias_f,94.770597
118,v8_provider_mean_remaining_vs_month_normal_f,94.375925
119,observed_heat_index_at_as_of_f,44.992600


## Train-Fold 3% Missingness Audit


In [9]:
modeling_frame, candidate_categorical, candidate_numeric = _modeling_frame(result.features, config)
candidate_features = [*candidate_categorical, *candidate_numeric]
missingness_rows = []
years = pd.to_numeric(modeling_frame["year"], errors="coerce")

for fold in [*V20_EXPANDING_FOLDS, ("test_refit_2021_2025", 2021, 2025)]:
    if isinstance(fold, tuple):
        fold_name, train_start, train_end = fold
    else:
        fold_name, train_start, train_end = fold.name, fold.train_start_year, fold.train_end_year
    train = modeling_frame.loc[years.between(train_start, train_end)].copy()
    retained_categorical, retained_numeric = _fit_feature_columns(
        train,
        candidate_categorical,
        candidate_numeric,
        max_missing_fraction=0.03,
    )
    retained = set(retained_categorical) | set(retained_numeric)
    for feature in candidate_features:
        missingness_rows.append(
            {
                "fold": fold_name,
                "feature": feature,
                "missing_fraction": float(train[feature].isna().mean()),
                "retained": feature in retained,
            }
        )

fold_feature_missingness = pd.DataFrame(missingness_rows)
fold_feature_missingness.to_csv(
    config.resolved_output_dir() / "HKO_fold_feature_missingness.csv",
    index=False,
)
fold_feature_missingness.loc[~fold_feature_missingness["retained"]].sort_values(
    ["fold", "missing_fraction"], ascending=[True, False]
)


,fold,feature,missing_fraction,retained
122,fold_2021_2022_to_2023,observed_weather_code_at_as_of,1.000000,False
137,fold_2021_2022_to_2023,observed_heat_index_at_as_of_f,0.576112,False
243,fold_2021_2023_to_2024,observed_weather_code_at_as_of,1.000000,False
258,fold_2021_2023_to_2024,observed_heat_index_at_as_of_f,0.557107,False
364,fold_2021_2024_to_2025,observed_weather_code_at_as_of,1.000000,False
379,fold_2021_2024_to_2025,observed_heat_index_at_as_of_f,0.540070,False
1,fold_2021_to_2022,observed_weather_code_at_as_of,1.000000,False
16,fold_2021_to_2022,observed_heat_index_at_as_of_f,0.530303,False
56,fold_2021_to_2022,gfs_prior_month_bias_f,0.090909,False
57,fold_2021_to_2022,gfs_prior_month_mae_f,0.090909,False


## Feature Importance


In [10]:
result.feature_importance.sort_values(
    ["method", "importance_mean_mae_f"],
    ascending=[True, False],
).groupby("method", as_index=False).head(20)


,method,param_key,feature,importance_mean_mae_f,importance_std_mae_f,n_repeats,train_start_year,train_end_year,test_year,train_rows,test_rows
2,catboost,trial_23,observed_high_temp_minus_temp_at_as_of_f,0.050530,0.011572,10,2021,2025,2026,1513,201
4,catboost,trial_23,observed_humidity_at_as_of,0.039990,0.011093,10,2021,2025,2026,1513,201
6,catboost,trial_23,v3_high_so_far_above_current_f,0.039053,0.008984,10,2021,2025,2026,1513,201
17,catboost,trial_23,v8_forecast_dewpoint_depression_mean_f,0.025168,0.013975,10,2021,2025,2026,1513,201
20,catboost,trial_23,v8_cloud_cover_mean_remaining_warmup_interaction,0.022375,0.011682,10,2021,2025,2026,1513,201
23,catboost,trial_23,v8_wind_gust_max_remaining_warmup_interaction,0.018793,0.010386,10,2021,2025,2026,1513,201
26,catboost,trial_23,v4_forecast_precip_hours_mean,0.014329,0.007241,10,2021,2025,2026,1513,201
28,catboost,trial_23,v11sf_forecast_warmup_after_11am_f,0.012724,0.009150,10,2021,2025,2026,1513,201
30,catboost,trial_23,v8_wind_speed_mean_remaining_warmup_interaction,0.011418,0.006388,10,2021,2025,2026,1513,201
36,catboost,trial_23,v4_precip_humidity_interaction,0.008733,0.005239,10,2021,2025,2026,1513,201


## Fahrenheit and Celsius Metrics


In [11]:
dual_metrics = add_celsius_metric_columns(result.metrics)
dual_scoreboard = add_celsius_metric_columns(result.scoreboard)
dual_metrics, dual_scoreboard


(        evaluation_scope       method  count     mae_f    rmse_f    bias_f  \
 0        year_split_test  ridge_stack    201  1.115964  1.403891  0.217174   
 1        year_split_test     catboost    201  1.130054  1.415981  0.153614   
 2        year_split_test     lightgbm    201  1.112866  1.422768  0.149907   
 3        year_split_test      xgboost    201  1.128274  1.435054  0.287999   
 4        year_split_test      gfs_raw    201  4.450877  4.976509  4.344765   
 5  year_split_validation     catboost   1447  1.155186  1.463264  0.012250   
 6  year_split_validation      xgboost   1447  1.162343  1.471225  0.013819   
 7  year_split_validation     lightgbm   1447  1.174450  1.502566  0.016885   
 8  year_split_validation      gfs_raw   1447  4.371660  4.960448  4.260746   
 
    p95_absolute_error_f  large_miss_5f_pct  within_1f_pct  within_2f_pct  ...  \
 0              2.724238           0.497512      49.751244      86.567164  ...   
 1              2.725516           0.000000 

## 2026 Celsius Buckets (Nearest Degree, Half-Up)


In [12]:
dual_test_predictions = add_celsius_prediction_columns(result.test_predictions)
c_bucket_predictions = result.bracket_predictions
c_bucket_metrics = result.bracket_metrics

assert c_bucket_metrics["bucket_unit"].eq("celsius").all()
assert c_bucket_metrics["bucket_width_c"].eq(1.0).all()
assert c_bucket_metrics["rounding_rule"].eq("floor_integer_celsius").all()
c_bucket_metrics


,method,bucket_unit,bucket_width_c,rounding_rule,count,exact_bucket_accuracy,exact_bucket_accuracy_pct,within_1c_accuracy,within_1c_accuracy_pct,two_bucket_accuracy,two_bucket_accuracy_pct,bucket_mae_c
0,catboost,celsius,1.0,floor_integer_celsius,201,0.452736,45.273632,0.960199,96.0199,0.651741,65.174129,0.59204
1,lightgbm,celsius,1.0,floor_integer_celsius,201,0.452736,45.273632,0.940299,94.029851,0.631841,63.18408,0.61194
2,xgboost,celsius,1.0,floor_integer_celsius,201,0.452736,45.273632,0.945274,94.527363,0.616915,61.691542,0.61194
3,ridge_stack,celsius,1.0,floor_integer_celsius,201,0.437811,43.781095,0.945274,94.527363,0.631841,63.18408,0.621891
4,gfs_raw,celsius,1.0,floor_integer_celsius,201,0.079602,7.960199,0.199005,19.900498,0.094527,9.452736,2.552239


## 2026 Monthly Metrics


In [13]:
monthly = dual_test_predictions.copy()
monthly["month"] = pd.to_datetime(monthly["contract_date"], errors="coerce").dt.month
monthly_metrics = (
    monthly.dropna(subset=["month", "error_f"])
    .groupby(["method", "month"], as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        rmse_f=("error_f", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_f=("error_f", "mean"),
        mae_c=("absolute_error_c", "mean"),
        rmse_c=("error_c", lambda values: float(np.sqrt(np.mean(np.square(values))))),
        bias_c=("error_c", "mean"),
    )
)
monthly_metrics.to_csv(config.resolved_output_dir() / "HKO_2026_monthly_metrics_dual_units.csv", index=False)
monthly_metrics


,method,month,count,mae_f,rmse_f,bias_f,mae_c,rmse_c,bias_c
0,catboost,1,31,0.818639,1.053217,0.102186,0.454799,0.585121,0.056770
1,catboost,2,28,1.370287,1.668538,0.251409,0.761270,0.926966,0.139672
2,catboost,3,31,1.160572,1.386043,0.503919,0.644762,0.770024,0.279955
3,catboost,4,30,1.162104,1.550887,-0.166115,0.645613,0.861604,-0.092286
4,catboost,5,31,1.120074,1.476425,0.028274,0.622263,0.820236,0.015708
5,catboost,6,30,1.213875,1.386632,-0.014562,0.674375,0.770351,-0.008090
6,catboost,7,20,1.070786,1.298000,0.479576,0.594881,0.721111,0.266431
7,gfs_raw,1,31,3.409296,3.800923,3.299586,1.894053,2.111624,1.833103
8,gfs_raw,2,28,4.809470,5.347505,4.766660,2.671928,2.970836,2.648145
9,gfs_raw,3,31,5.360624,5.914816,5.341347,2.978124,3.286009,2.967415


## GFS Raw Uplift


In [14]:
test_mae = (
    dual_test_predictions.groupby("method", as_index=False)
    .agg(count=("absolute_error_f", "size"), mae_f=("absolute_error_f", "mean"), mae_c=("absolute_error_c", "mean"))
)
gfs_mae_f = float(test_mae.loc[test_mae["method"].eq("gfs_raw"), "mae_f"].iloc[0])
gfs_mae_c = float(test_mae.loc[test_mae["method"].eq("gfs_raw"), "mae_c"].iloc[0])
test_mae["mae_uplift_vs_gfs_f"] = gfs_mae_f - test_mae["mae_f"]
test_mae["mae_uplift_vs_gfs_c"] = gfs_mae_c - test_mae["mae_c"]
test_mae.sort_values("mae_f")


,method,count,mae_f,mae_c,mae_uplift_vs_gfs_f,mae_uplift_vs_gfs_c
2,lightgbm,201,1.112866,0.618259,3.338011,1.854450
3,ridge_stack,201,1.115964,0.619980,3.334913,1.852729
4,xgboost,201,1.128274,0.626819,3.322603,1.845891
0,catboost,201,1.130054,0.627808,3.320823,1.844901
1,gfs_raw,201,4.450877,2.472709,0.000000,0.000000


## Performance by Warm/Cool 11 AM Forecast Delta


In [15]:
delta_by_date = result.features[
    ["contract_date", "v11sf_forecast_temp_11am_minus_observed_f"]
].copy()
delta_by_date["forecast_temp_delta_c"] = (
    pd.to_numeric(delta_by_date["v11sf_forecast_temp_11am_minus_observed_f"], errors="coerce") * 5 / 9
)
delta_predictions = dual_test_predictions.merge(delta_by_date, on="contract_date", how="left")
delta_predictions["forecast_temp_delta_bucket"] = pd.cut(
    delta_predictions["forecast_temp_delta_c"],
    bins=[-np.inf, -1.0, -0.25, 0.25, 1.0, np.inf],
    labels=["cool_gt_1c", "cool_0.25_to_1c", "near_match", "warm_0.25_to_1c", "warm_gt_1c"],
)
warm_cool_metrics = (
    delta_predictions.dropna(subset=["forecast_temp_delta_bucket", "error_f"])
    .groupby(["method", "forecast_temp_delta_bucket"], observed=True, as_index=False)
    .agg(
        count=("error_f", "size"),
        mae_f=("absolute_error_f", "mean"),
        bias_f=("error_f", "mean"),
        mae_c=("absolute_error_c", "mean"),
        bias_c=("error_c", "mean"),
    )
)
warm_cool_metrics.to_csv(config.resolved_output_dir() / "HKO_warm_cool_delta_metrics.csv", index=False)
warm_cool_metrics


,method,forecast_temp_delta_bucket,count,mae_f,bias_f,mae_c,bias_c
0,catboost,cool_gt_1c,125,1.102844,0.062320,0.612691,0.034622
1,catboost,cool_0.25_to_1c,32,1.187996,0.566015,0.659998,0.314453
2,catboost,near_match,14,1.281632,0.672265,0.712018,0.373481
3,catboost,warm_0.25_to_1c,19,0.969496,-0.276091,0.538609,-0.153384
4,catboost,warm_gt_1c,11,1.355117,0.073446,0.752843,0.040803
5,gfs_raw,cool_gt_1c,125,5.368215,5.351427,2.982342,2.973015
6,gfs_raw,cool_0.25_to_1c,32,3.762259,3.721555,2.090144,2.067531
7,gfs_raw,near_match,14,3.639647,3.554028,2.022026,1.974460
8,gfs_raw,warm_0.25_to_1c,19,1.839534,1.717193,1.021963,0.953996
9,gfs_raw,warm_gt_1c,11,1.572806,0.263318,0.873781,0.146288
